In [2]:
from sympy import pprint

from triplet_extraction.src.db.mongo import init_mongo

mongo = init_mongo()
db = mongo["KB_PROPERTY_LAW"]

documents_col = db["documents"]
sections_col = db["legal_sections"]
section_relations_col = db["legal_section_relations"]

You successfully connected to MongoDB!


In [3]:
from triplet_extraction.src.db import build_tree_downward, print_tree

doc = sections_col.find({
    "is_amendment": True,
    "type": "điều"
})

count = 0

for d in doc:
    count += 1

    print("\n" + "=" * 80)
    print(f"Processing: {d['full_path']}")
    print("=" * 80)

    downward_tree = build_tree_downward(sections_col, d["_id"])
    print_tree(downward_tree, show_content=False)
    print("\n")

print(f"Total amendment articles processed: {count}")


Processing: 31/2024/QH15_chương xvi_mục 1_điều 243
├─ [điều] điều 243
  ├─ [khoản] khoản 1
  ├─ [khoản] khoản 2
    ├─ [điểm] điểm a
    ├─ [điểm] điểm b
    ├─ [điểm] điểm c
  ├─ [khoản] khoản 3
  ├─ [khoản] khoản 4



Processing: 31/2024/QH15_chương xvi_mục 1_điều 244
├─ [điều] điều 244



Processing: 31/2024/QH15_chương xvi_mục 1_điều 245
├─ [điều] điều 245
  ├─ [khoản] khoản 1
  ├─ [khoản] khoản 2
  ├─ [khoản] khoản 3
  ├─ [khoản] khoản 4
  ├─ [khoản] khoản 5



Processing: 31/2024/QH15_chương xvi_mục 1_điều 246
├─ [điều] điều 246



Processing: 31/2024/QH15_chương xvi_mục 1_điều 247
├─ [điều] điều 247



Processing: 31/2024/QH15_chương xvi_mục 1_điều 248
├─ [điều] điều 248
  ├─ [khoản] khoản 1
  ├─ [khoản] khoản 2
  ├─ [khoản] khoản 3
  ├─ [khoản] khoản 4
  ├─ [khoản] khoản 5
  ├─ [khoản] khoản 6
    ├─ [điểm] điểm a
    ├─ [điểm] điểm b
  ├─ [khoản] khoản 7
    ├─ [điểm] điểm a
    ├─ [điểm] điểm b
  ├─ [khoản] khoản 8
    ├─ [điểm] điểm a
    ├─ [điểm] điểm b
  ├─ [khoản] khoản

In [4]:
from triplet_extraction.src.db import build_tree_downward, print_tree

doc = sections_col.find({
    "is_amendment": True,
    "type": "điều"
})

count = 0

for d in doc:
    downward_tree = build_tree_downward(sections_col, d["_id"])
    # if len(downward_tree['children']) > 0:
    #     continue
    count += 1
    print("\n" + "=" * 80)
    print(f"Processing: {d['full_path']}")
    print("=" * 80)

    pprint(downward_tree)

print(f"Total amendment articles processed: {count}")


Processing: 31/2024/QH15_chương xvi_mục 1_điều 243
{_id: 503765f0cec0d6111a8ea9bac43ebf1c0dbb404a951e72ce538ed8e0569de7b1, childr ↪

↪ en: [{_id: a40857942fc9c1ad008596d21dc393ba37d6c3b441b4dcb81c243006cddf3756, ↪

↪  children: [], content: Sửa đổi, bổ sung khoản 2 Điều 24 như sau: “2. Quy ho ↪

↪ ạch sử dụng đất quốc gia bao gồm những nội dung chủ yếu sau đây: a) Phân tíc ↪

↪ h, đánh giá về các yếu tố, điều kiện tự nhiên, nguồn lực, bối cảnh trực tiếp ↪

↪  tác động và thực trạng sử dụng đất của các ngành, lĩnh vực; b) Dự báo xu th ↪

↪ ế biến động của việc sử dụng đất; c) Xác định các quan điểm và mục tiêu sử d ↪

↪ ụng đất trong thời kỳ mới; d) Định hướng sử dụng đất quốc gia, vùng kinh tế  ↪

↪ - xã hội, tầm nhìn sử dụng đất đáp ứng nhu cầu sử dụng đất để phát triển kin ↪

↪ h tế - xã hội; bảo đảm quốc phòng, an ninh; bảo vệ môi trường, thích ứng với ↪

↪  biến đổi khí hậu; đ) Xác định các chỉ tiêu sử dụng đất đối với nhóm đất nôn ↪

↪ g nghiệp, nhóm đất phi nông nghiệp; trong đó

KeyboardInterrupt: 

In [25]:
import re
from typing import Optional, List

KHOAN_PATTERN = re.compile(r"khoản\s+(\d+)", re.IGNORECASE)
DIEU_PATTERN = re.compile(r"điều\s+(\d+)", re.IGNORECASE)
DIEM_PATTERN = re.compile(r"điểm\s+([a-z])", re.IGNORECASE)
PHAN_PATTERN = re.compile(r"phần\s+thứ\s+([ivxlcdm\d]+)", re.IGNORECASE)
CHUONG_PATTERN = re.compile(r"chương\s+([ivxlcdm\d]+)", re.IGNORECASE)
MUC_PATTERN = re.compile(r"mục\s+([ivxlcdm\d]+)", re.IGNORECASE)
TIEU_MUC_PATTERN = re.compile(r"tiểu\s+mục\s+(\d+)", re.IGNORECASE)
PHU_LUC_PATTERN = re.compile(r"phụ\s+lục(?:\s+([ivxlcdm\d]+))?", re.IGNORECASE)

# Enhanced patterns to support multiple document types
SO_HIEU_PATTERN = re.compile(
    r"số\s+(\d+/\d+(?:/[a-z\-]+)?)",  # Matches: 40/2024/TT-BGTVT, 68/2014/NĐ-CP, 12/2020/QH14, etc.
    re.IGNORECASE
)

# Enhanced law/document name pattern to capture various legal document types
DOC_NAME_PATTERN = re.compile(
    r"((?:Bộ\s+)?(?:Luật|Nghị định|Thông tư|Quyết định|Chỉ thị|Nghị quyết)\s+[^\d\n]+?)(?=\s+số|\s+năm|$)",
    re.IGNORECASE
)


def fuzzy_search_so_hieu(document_col, doc_name: str) -> Optional[str]:
    """
    Fuzzy search for so_hieu in document collection using document name.
    Uses text search with case-insensitive regex.
    """
    # Try exact match first
    doc = document_col.find_one(
        {"title": {"$regex": f"^{re.escape(doc_name)}$", "$options": "i"}},
        {"so_hieu": 1}
    )

    if doc:
        return doc.get("so_hieu")

    # Try partial match
    doc = document_col.find_one(
        {"title": {"$regex": re.escape(doc_name), "$options": "i"}},
        {"so_hieu": 1}
    )

    if doc:
        return doc.get("so_hieu")

    return None


def parse_amendment_reference(text: str, document_col) -> dict | None:
    """
    Extract all available legal reference info from text.
    - Only take the FIRST occurrence of each component
    - Missing components are returned as None
    - If nothing is found at all, return None
    - If so_hieu is missing but doc_name exists and document_col is provided,
      attempts to resolve so_hieu via fuzzy search
    """
    text = text.split(":")[0]  # Consider only text before first colon

    khoan_m = KHOAN_PATTERN.search(text)
    dieu_m = DIEU_PATTERN.search(text)
    diem_m = DIEM_PATTERN.search(text)
    phan_m = PHAN_PATTERN.search(text)
    chuong_m = CHUONG_PATTERN.search(text)
    muc_m = MUC_PATTERN.search(text)
    tieu_muc_m = TIEU_MUC_PATTERN.search(text)
    phu_luc_m = PHU_LUC_PATTERN.search(text)
    so_hieu_m = SO_HIEU_PATTERN.search(text)
    doc_name_m = DOC_NAME_PATTERN.search(text)

    if not (khoan_m or dieu_m or diem_m or phan_m or chuong_m or muc_m or
            tieu_muc_m or phu_luc_m or so_hieu_m or doc_name_m):
        return None

    so_hieu = so_hieu_m.group(1).upper() if so_hieu_m else None
    doc_name = doc_name_m.group(1).strip() if doc_name_m else None

    # If so_hieu is missing but doc_name exists, try fuzzy search
    if not so_hieu and doc_name:
        so_hieu = fuzzy_search_so_hieu(document_col, doc_name)

    return {
        "phan": phan_m.group(1).lower() if phan_m else None,
        "chuong": chuong_m.group(1).lower() if chuong_m else None,
        "muc": muc_m.group(1).lower() if muc_m else None,
        "tieu_muc": tieu_muc_m.group(1) if tieu_muc_m else None,
        "phu_luc": phu_luc_m.group(1).lower() if phu_luc_m and phu_luc_m.group(1) else None,
        "dieu": dieu_m.group(1) if dieu_m else None,
        "khoan": khoan_m.group(1) if khoan_m else None,
        "diem": diem_m.group(1).lower() if diem_m else None,
        "so_hieu": so_hieu,
        "doc_name": doc_name,  # Changed from law_name to doc_name for clarity
    }


def parse_amendment_type(text: str) -> List[str]:
    """
    Parse amendment type(s) from text.
    A sentence can have multiple types (e.g., "sửa đổi, bổ sung").

    Args:
        text: Text to parse for amendment types

    Returns:
        List of amendment types found, in order of appearance
    """
    AMENDMENT_PATTERNS = {
        'modify': [
            r'sửa đổi',
            r'đổi',
        ],
        'add': [
            r'bổ sung',
            r'thêm',
            r'tăng',
        ],
        'remove': [
            r'bãi bỏ',
            r'xóa bỏ',
            r'hủy bỏ',
            r'loại bỏ',
        ],
        'replace': [
            r'thay thế',
            r'thay',
        ]
    }

    text = text.split(":")[0]  # Consider only text before first colon

    found_types = []
    seen_types = set()

    # Create a list of (position, type) tuples for all matches
    matches = []

    for amendment_type, patterns in AMENDMENT_PATTERNS.items():
        for pattern in patterns:
            for match in re.finditer(pattern, text, re.IGNORECASE):
                matches.append((match.start(), amendment_type))

    # Sort by position to maintain order of appearance
    matches.sort(key=lambda x: x[0])

    # Add types in order, avoiding duplicates
    for _, amendment_type in matches:
        if amendment_type not in seen_types:
            found_types.append(amendment_type)
            seen_types.add(amendment_type)

    return found_types


def resolve_full_path(sections_col, ref: dict) -> str | None:
    """
    Resolve amendment reference to canonical full_path using MongoDB.
    """
    # If no so_hieu or dieu, cannot resolve
    if not ref.get("so_hieu") or not ref.get("dieu"):
        return None

    dieu_title = f"điều {ref['dieu']}"

    node = sections_col.find_one(
        {
            "so_hieu": ref["so_hieu"].upper(),
            "type": "điều",
            "title": {"$regex": f"^{dieu_title}$", "$options": "i"}
        },
        {"full_path": 1}
    )

    if not node:
        return None

    return node["full_path"]

In [26]:
text = "thay thế một số nội dung của thông tư số 40/2024/tt-bgtvt ngày 15 tháng 11 năm 2024 của bộ trưởng bộ giao thông vận tải quy định về công tác phòng, chống, khắc phục hậu quả thiên tai trong lĩnh vực đường bộ"

ref = parse_amendment_reference(text, documents_col)
amendment_type = parse_amendment_type(text)
print(ref)
print(amendment_type)
if ref:
    full_path = resolve_full_path(sections_col, ref)
    print(full_path)

{'phan': None, 'chuong': None, 'muc': None, 'tieu_muc': None, 'phu_luc': None, 'dieu': None, 'khoan': None, 'diem': None, 'so_hieu': '40/2024/TT-BGTVT', 'doc_name': None}
['replace']
None


In [6]:
from typing import Dict, Optional, Set

def print_tree_with_ref(
    node: Dict,
    ref: Optional[Dict[str, str]] = None,
    indent: int = 0,
    show_content: bool = False,
    visited: Optional[Set[int]] = None
) -> None:
    """
    Pretty print the tree structure with amendment references.

    Args:
        node: Tree node containing type, title, content, and optional children/parent
        ref: Reference dictionary with keys like 'so_hieu', 'dieu', 'khoan'
        indent: Current indentation level
        show_content: Whether to show content preview
        visited: Set of visited node IDs to prevent infinite recursion
    """
    if not node:
        return

    # Prevent infinite recursion
    if visited is None:
        visited = set()

    node_id = id(node)
    if node_id in visited:
        prefix = "  " * indent
        print(f"{prefix}├─ [CIRCULAR REFERENCE]")
        return
    visited.add(node_id)

    # Parse new reference from current node
    new_ref = parse_amendment_reference(node['content'])

    # Initialize ref if None
    if ref is None:
        ref = new_ref or {}

    # Merge references: copy ref and update with new_ref values
    current_ref = ref.copy() if ref else {}
    if new_ref:
        for key in ['so_hieu', 'dieu', 'khoan']:
            if new_ref.get(key):
                current_ref[key] = new_ref[key]

    # Print node information
    prefix = "  " * indent
    print(f"{prefix}├─ [{node['type']}] {node['title']} - {node['full_path']}")
    print(f"{prefix}    {current_ref}")

    if show_content and node.get('content'):
        content_preview = node['content'][:100] + "..." if len(node['content']) > 100 else node['content']
        print(f"{prefix}   Content: {content_preview}")

    # Print children if using downward tree
    if 'children' in node:
        for child in node['children']:
            print_tree_with_ref(child, current_ref, indent + 1, show_content, visited)

    # Print parent if using upward tree
    if 'parent' in node:
        print(f"{prefix}   ↑ Parent:")
        print_tree_with_ref(node['parent'], current_ref, indent + 1, show_content, visited)

In [23]:
from typing import Dict, Optional, Set, Tuple, List

def add_amendment_ref_to_nodes(
    node: Dict,
    documents_col,
    ref: Optional[Dict[str, str]] = None,
    verbose: bool = False,
) -> List[Dict]:
    """
    Add amendment references to each node in the tree structure using iterative approach.
    Traverses downward through children only. Returns array of unique leaf nodes.
    Only adds ref properties if they don't already exist in the node's ref.
    """
    if not node:
        return []

    # Stack contains tuples of (node, current_ref)
    stack: list[Tuple[Dict, Dict[str, str]]] = [(node, ref or {})]
    visited: Set[int] = set()
    leaf_nodes: List[Dict] = []
    seen_paths: Set[str] = set()  # Track unique full_paths

    while stack:
        current_node, current_ref = stack.pop()

        # Prevent infinite recursion
        node_id = id(current_node)
        if node_id in visited:
            continue
        visited.add(node_id)

        if current_node.get('is_amendment') is not True:
            continue

        # Parse new reference from current node
        new_ref = parse_amendment_reference(current_node['content'], documents_col)

        # Merge references: copy ref and update with new_ref values
        merged_ref = current_ref.copy()
        if new_ref:
            for key in ['so_hieu', 'dieu', 'khoan', 'diem', 'law_name',
                       'phan', 'chuong', 'muc', 'tieu_muc', 'phu_luc']:
                if new_ref.get(key) and not merged_ref.get(key):
                    merged_ref[key] = new_ref[key]

        # Initialize ref dict if it doesn't exist
        if 'ref' not in current_node:
            current_node['ref'] = {}

        # Only add individual properties if they don't exist
        for key in ['so_hieu', 'dieu', 'khoan', 'diem', 'law_name',
                   'phan', 'chuong', 'muc', 'tieu_muc', 'phu_luc']:
            if key not in current_node['ref'] and key in merged_ref:
                current_node['ref'][key] = merged_ref[key]

        amendment_type = parse_amendment_type(current_node['content'])
        current_node['ref']['amendment_type'] = amendment_type

        # Check if this is a leaf node (no children or empty children list)
        is_leaf = 'children' not in current_node or not current_node['children']

        if is_leaf:
            full_path = current_node.get('full_path')

            # Only add if we haven't seen this full_path before
            if full_path and full_path not in seen_paths:
                leaf_nodes.append(current_node)
                seen_paths.add(full_path)
                if verbose:
                    print(f"Leaf: {full_path}")
                    print(f"Ref: {current_node.get('ref', {})}")
                    print()
            elif full_path in seen_paths:
                if verbose:
                    print(f"Skipping duplicate: {full_path}")

        # Add children to stack
        if 'children' in current_node:
            for child in current_node['children']:
                stack.append((child, merged_ref))

    return leaf_nodes


# Usage example
from triplet_extraction.src.db import build_tree_downward, print_tree

amendment_articles = sections_col.find({
    "is_amendment": True,
    "type": "điều"
})

count = 0
all_leaves = []
seen_global_paths: Set[str] = set()  # Track across all articles

for article in amendment_articles:
    count += 1

    print(f"Processing: {article['full_path']}")
    downward_tree = build_tree_downward(sections_col, article["_id"])
    leaf_nodes = add_amendment_ref_to_nodes(downward_tree, documents_col, None, True)

    # Only add leaves that haven't been seen globally
    for leaf in leaf_nodes:
        full_path = leaf.get('full_path')
        if full_path and full_path not in seen_global_paths:
            all_leaves.append(leaf)
            seen_global_paths.add(full_path)

    print(f"Found {len(leaf_nodes)} unique leaf nodes in this article")
    print("\n")

# Now you can work with all_leaves array (all unique)
for leaf in all_leaves:
    print(f"{leaf['full_path']}: {leaf.get('ref', {})}")

print(f"Total amendment articles processed: {count}")
print(f"Total unique leaf nodes found: {len(all_leaves)}")

Processing: 31/2024/QH15_chương xvi_mục 1_điều 243
Leaf: 31/2024/QH15_chương xvi_mục 1_điều 243_khoản 4
Ref: {'so_hieu': '21/2017/QH14', 'law_name': 'luật quy hoạch', 'muc': 'c', 'phu_luc': 'ii', 'amendment_type': ['add']}

Leaf: 31/2024/QH15_chương xvi_mục 1_điều 243_khoản 3
Ref: {'so_hieu': '21/2017/QH14', 'dieu': '27', 'khoan': '2', 'diem': 'l', 'law_name': 'luật quy hoạch', 'amendment_type': ['modify', 'add']}

Leaf: 31/2024/QH15_chương xvi_mục 1_điều 243_khoản 2_điểm c
Ref: {'so_hieu': '21/2017/QH14', 'dieu': '25', 'khoan': '7', 'law_name': 'luật quy hoạch', 'amendment_type': ['modify', 'add']}

Leaf: 31/2024/QH15_chương xvi_mục 1_điều 243_khoản 2_điểm b
Ref: {'so_hieu': '21/2017/QH14', 'dieu': '25', 'khoan': '4', 'law_name': 'luật quy hoạch', 'amendment_type': ['add']}

Leaf: 31/2024/QH15_chương xvi_mục 1_điều 243_khoản 2_điểm a
Ref: {'so_hieu': '21/2017/QH14', 'dieu': '25', 'khoan': '4', 'law_name': 'luật quy hoạch', 'amendment_type': ['modify']}

Leaf: 31/2024/QH15_chương xvi_m